# Homework 8: Markov Processes

**Note.** This homework has un-numbered exercises intended for further exploration (not graded). Numbered exercises are collected in Canvas as usual.

## 8.0 Introduction

This homework accompanies Lecture 8 on linear dynamical systems and Markov processes. Rather than continuing to explore the SIRD epidemic model from lecture, we apply the same mathematical framework to a different domain: **pharmaco-kinetics**, the study of how drug concentrations change over time in the body.

The model is a compartmental system — a special case of a linear dynamical system $\mathbf{x}_{t+1} = A\mathbf{x}_t$ — where material moves between compartments (organs) and is gradually eliminated. Unlike the SIRD model, where the transition matrix is column-stochastic and the system has a non-trivial steady state, the drug model loses material at every step. As you will see, this is reflected directly in the eigenvalues of the transition matrix.

**Sections:**
- **8.1** Build and simulate the compartmental drug model
- **8.2** Convergence and limits — half-life, long-run behavior, and a safety check
- **8.3** Eigenvalue analysis — connecting eigenvalues to the decay rate
- **8.4** Recirculation — modifying the model and comparing eigenvalues
- **8.5** Epidemic model with vaccination — revisiting SIRD with eigenvalue analysis
- **8.6** Epidemic model with vaccination and immigration — affine dynamics

**How to work through this notebook.** Run every cell in order before moving to the next — later cells depend on variables defined in earlier ones. Cells marked `# Run this cell without modification` are scaffolding; read them carefully but do not change them. Cells marked `# Your code here` are yours to complete.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 8.1 A Linear Dynamical System: Pharmaco-Kinetics

The following model is adapted from *Introduction to Applied Linear Algebra* by Boyd and Vandenberghe (2018).

A **compartmental system** describes the movement of material over time among a set of compartments and the outside world. In pharmaco-kinetics, the material is a drug and the compartments are organs (bloodstream, liver, kidneys, etc.). Compartmental systems are special cases of linear dynamical systems.

We consider a system with three compartments. Let $\mathbf{x}_t$ denote the amount of drug (in milligrams) in the system at time $t$ (in hours). Between period $t$ and $t+1$, the drug moves as follows:

- 10% of the drug in Organ 1 moves to Organ 2.
- 5% of the drug in Organ 2 moves to Organ 3.
- 5% of the drug in Organ 3 recirculates back to Organ 1.
- 5% of the drug in Organ 3 is eliminated from the body.
- All remaining drug stays in its current organ.

This defines a linear dynamical system $\mathbf{x}_{t+1} = A\mathbf{x}_t$.

**Exercise.** Suppose 10 milligrams of a medication is injected directly into Organ 1 at $t = 0$, with none in Organs 2 or 3. Define the initial state vector $\mathbf{x}_0$.

In [ ]:
# Your code here
x_0 = 


**Exercise.** Construct the transition matrix $A$.

In [ ]:
# Your code here
A =  

**Exercise 1.** How many mg of the drug are in Organ 2 at 3 hours? 

*Hint:* `np.linalg.matrix_power(A, t)` computes $A^t$ without using a for-loop like we did in class. Feel free to use it here and throughout the homework.

In [ ]:
# Your code here


**Exercise 2.** How many milligrams of the medication are left in the body at 10 hours?

In [ ]:
# Your code here


Run the following cell to simulate and plot all three organ concentrations over 200 hours.

In [ ]:
# Run this cell without modification
T = 200
labels_pk = ['Organ 1', 'Organ 2', 'Organ 3']
colors_pk  = ['steelblue', 'tomato', 'seagreen']

history_pk = np.zeros((3, T+1))
history_pk[:, 0] = x_0
x = x_0.copy()
for t in range(T):
    x = A @ x
    history_pk[:, t+1] = x

fig, ax = plt.subplots(figsize=(9, 4))
for i in range(3):
    ax.plot(np.arange(T+1), history_pk[i], color=colors_pk[i], lw=2, label=labels_pk[i])
ax.set_xlabel('Hours')
ax.set_ylabel('Milligrams')
ax.set_title('Drug Concentration by Organ Over Time')
ax.legend()
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

## 8.2 Convergence and Limits

### Limits

Recall from Calculus 1: the **limit** of a sequence is the value it approaches as $t \to \infty$, getting arbitrarily close but not necessarily reaching it in finite time. For a drug model, the central question is whether the medication clears from the body — and if so, how quickly.

**Exercise 3.** Find the approximate **half-life** of the medication — the value of $t$ at which the total drug remaining in the body is closest to half the initial dose.

*Hint:* Use `np.linalg.matrix_power` and `sum()` to guess and check different values of $t$. The total will never equal exactly half the initial dose, so find the closest $t$.

In [ ]:
# Your code here


**Exercise 4.** How many hours does it take for the medication in Organ 1 to fall to 3 mg or less?

In [ ]:
# Your code here


**Exercise 5.** Suppose that at $t = 50$ an additional 5 mg is injected directly into Organ 1. How much total drug remains in the body at $t = 100$?

In [ ]:
# Your code here


**Exercise 6.** $\displaystyle\lim_{t \to \infty} \mathbf{x}_t = $ \_\_\_. Does this make sense in the context of this problem? How does this differ from the long-run results we saw in the SIRD model?

In [ ]:
# Your code here


Run the following cell to plot Organ 3 concentration over 1000 hours. How does this relate to the previous plot?

In [ ]:
# Run this cell without modification
T_long = 1000
organ3_vals = np.zeros(T_long + 1)
organ3_vals[0] = x_0[2]
x = x_0.copy()
for t in range(T_long):
    x = A @ x
    organ3_vals[t+1] = x[2]

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(np.arange(T_long + 1), organ3_vals, color='seagreen', lw=1.5)
ax.set_xlabel('Hours')
ax.set_ylabel('Milligrams')
ax.set_title('Drug Concentration in Organ 3 Over Time')
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

### Safety Mechanisms in Modern Medicine

Hospitals increasingly use software to catch drug dosage errors before they happen. Consider two safety constraints for Organ 3:

- **Peak constraint:** the concentration in Organ 3 must never exceed 3 mg (above this level, toxic side effects occur).
- **Cumulative exposure constraint:** the total drug exposure of Organ 3 over all time — measured in mg·hrs — must not exceed 150 mg·hrs (prolonged exposure also causes toxicity).

For those who have taken Calculus 2: the cumulative exposure is the improper integral $\int_0^\infty f(t)\,dt$, where $f(t)$ is the Organ 3 concentration at time $t$. We can use `np.trapezoid` to numerically estimate this integral from our simulated values.

Run the following cell to define the safety check function. Notice that the function parameters are named `transition_matrix` and `initial_dose` — these are local names inside the function and do not need to match whatever variable names you use when calling it.

In [ ]:
# Run this cell without modification
def safety_check(transition_matrix, initial_dose):
    y = np.zeros(1001)
    y[0] = initial_dose[2]
    x = initial_dose.copy().astype(float)
    for i in range(1, 1001):
        x = transition_matrix @ x
        y[i] = x[2]
    if np.max(y) > 3:
        return f'Warning: peak Organ 3 concentration = {np.round(np.max(y), 4)} mg (limit: 3 mg).'
    elif np.trapezoid(y) > 150:
        return f'Warning: cumulative Organ 3 exposure = {np.round(np.trapezoid(y), 4)} mg·hrs (limit: 150 mg·hrs).'
    else:
        return 'Initial dose is safe.'

**Exercise.** Use `safety_check` for the 10 mg initial dose. Which constraint is violated, if any?

In [ ]:
# Your code here


## 8.3 Recirculation Modification

In the base model, 5% of the drug in Organ 3 recirculates to Organ 1 and 5% is eliminated each hour. Now suppose a drug formulation change causes more of the drug to recirculate rather than be eliminated:

- **8%** of the drug in Organ 3 recirculates to Organ 1 (up from 5%).
- **2%** of the drug in Organ 3 is eliminated (down from 5%).
- All other rates remain unchanged.

**Exercise.** Construct the modified transition matrix `A_recirc`. What should each column sum to, and why? Verify by printing the column sums.

In [ ]:
# Your code here
A_recirc = 

Run the following cell to plot the total drug remaining in the body over time for both models.

In [ ]:
# Run this cell without modification
T_comp = 500
total_base   = np.zeros(T_comp + 1)
total_recirc = np.zeros(T_comp + 1)
total_base[0]   = x_0.sum()
total_recirc[0] = x_0.sum()

x_b = x_0.copy()
x_r = x_0.copy()
for t in range(T_comp):
    x_b = A        @ x_b
    x_r = A_recirc @ x_r
    total_base[t+1]   = x_b.sum()
    total_recirc[t+1] = x_r.sum()

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(np.arange(T_comp+1), total_base,   color='steelblue', lw=2, label='Base model')
ax.plot(np.arange(T_comp+1), total_recirc, color='tomato',    lw=2, label='Recirculation model')
ax.axhline(5, color='gray', lw=1, linestyle='--', label='Half of initial dose')
ax.set_xlabel('Hours')
ax.set_ylabel('Total drug in body (mg)')
ax.set_title('Total Drug Remaining: Base vs. Recirculation')
ax.legend()
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

## 8.4 Epidemic Dynamics Model with Vaccination

We now return briefly to the SIRD model from Lecture 8. Recall the base transition matrix with compartments $S$, $I$, $R$, $D$ and transition rates: 5% S→I, 4% I→S, 10% I→R, 1% I→D. Its eigenvalues are $1,\ 1,\ 0.9671,\ 0.8329$.

Now suppose a vaccine is available at $t = 0$: at every time step, 2.5% of susceptible individuals are vaccinated and move directly to Recovered.

**Exercise.** Construct both `A_sird` (the base SIRD matrix from lecture) and `A_vax` (the vaccination modification). Verify that both are column-stochastic (add to 1, no "leakage" from the system).

In [ ]:
# Your code here
A_sird = np.array([
    [0.95, 0.04, 0,    0],
    [0.05, 0.85, 0,    0],
    [0,    0.10, 1,    0],
    [0,    0.01, 0,    1],
])

A_vax = 

print('A_sird column sums:', np.round(A_sird.sum(axis=0), 4))
print('A_vax  column sums:', np.round(A_vax.sum(axis=0),  4))

Run the following cell to simulate and plot both models. We start from a fully susceptible population — since 5% of susceptible individuals become infected each step regardless, the epidemic gets going on its own without needing anyone infected at $t = 0$.

In [ ]:
# Run this cell without modification
x_0_sird    = np.array([1.0, 0.0, 0.0, 0.0])
labels_sird = ['Susceptible', 'Infected', 'Recovered', 'Deceased']
colors_sird = ['steelblue', 'tomato', 'seagreen', '#888888']

T_sird = 300
history_sird = np.zeros((4, T_sird+1))
history_vax  = np.zeros((4, T_sird+1))
history_sird[:, 0] = x_0_sird
history_vax[:,  0] = x_0_sird

x_s = x_0_sird.copy()
x_v = x_0_sird.copy()
for t in range(T_sird):
    x_s = A_sird @ x_s
    x_v = A_vax  @ x_v
    history_sird[:, t+1] = x_s
    history_vax[:,  t+1] = x_v

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (hist, title) in zip(axes, [
    (history_sird, 'Base SIRD (no vaccine)'),
    (history_vax,  'SIRD with Vaccination (2.5% S→R per step)'),
]):
    for i in range(4):
        ax.plot(np.arange(T_sird+1), hist[i], color=colors_sird[i], lw=2, label=labels_sird[i])
    ax.set_xlabel('Time step $t$')
    ax.set_ylabel('Fraction of population')
    ax.set_title(title)
    ax.legend(fontsize=9)
    ax.set_xlim(0, T_sird); ax.set_ylim(0, 1)
    ax.grid(True, linestyle='--', alpha=0.4)
plt.suptitle('SIRD Model: Base vs. Vaccination', fontsize=13)
plt.tight_layout()
plt.show()

**Exercise 7.** Starting from a fully susceptible population, what is the difference in percentage of the population is deceased at $t = 1000$? when comparing the vaccinated and un-vaccinated populations? Make sure you format your answer as a percentage.

In [ ]:
# Your code here


**Exercise.** Compute the eigenvalues for `A_vax`. 

We didn't analyze this in detail in class, but it is well-know that the non $\lambda = 1$ eigenvalues control the rate at which the population approaches its long-run steady-state: smaller eigenvalues indicate faster convergence for the system. Think (intuitively) about what a vaccince will do to the speed of convergence. Does this intuition align with what you see when you compare the eigenvalues $A_{sird}$ (from class) and $A_{vax}$?




In [ ]:
# Your code here


## 8.4 Epidemic Model with Vaccination and Immigration

We now extend the vaccination model to include immigration. The update equation becomes:

$$\mathbf{x}_{t+1} = A_{\text{vax}}\,\mathbf{x}_t + \mathbf{v}_t$$

where $\mathbf{v}_t$ is an immigration vector at time $t$. This is called an **affine** dynamical system — the pure linear structure of the Markov model is broken by the additive term.

The immigration policy:
- A fixed 0.5% of the initial population (i.e., 0.005 per step) immigrates each time step.
- No infected or deceased individuals may immigrate — only susceptible or recovered.
- At each time step, a fraction $a \sim \text{Uniform}[0, 1]$ of the immigrants are susceptible; the remaining fraction $(1-a)$ are recovered.

**Exercise.** Derive expressions for $\mathbf{x}_1$, $\mathbf{x}_2$, and $\mathbf{x}_3$ in terms of $A_{\text{vax}}$, $\mathbf{x}_0$, $\mathbf{v}_0$, $\mathbf{v}_1$, $\mathbf{v}_2$. What does a general expression for $\mathbf{x}_n$ look like? *(Do this by hand.)*

### The Uniform Distribution

The $\text{Uniform}[0,1]$ distribution generates random values between 0 and 1, all equally likely. We use `np.random.default_rng(seed)` to create a reproducible random number generator — this ensures results are identical across machines and Python versions. Run the cell below a few times to see that not setting the seed generates different random values, but setting the seed to `42` always generates the same "random" sequence. 

In [ ]:
import numpy as np

# Without a seed — different every time you run this cell
rng = np.random.default_rng()
print("Without seed:", rng.random(6))

# With a seed — always the same result
rng_seeded = np.random.default_rng(42)
print("With seed 42:", rng_seeded.random(6))

Here's how this works with the $\text{Uniform}[0,1]$ distribution. 

In [ ]:
# Run this cell without modification
rng_example = np.random.default_rng()
a = rng_example.uniform(0, 1)
print(f'a     = {np.round(a, 4)}')
print(f'1 - a = {np.round(1 - a, 4)}')

**Exercise 8.** In the cell above, set the seed to `5` and report the value of `a`. 

**Exercise.** Suppose $a$ is drawn at time $t$ as above. Write an expression for the immigration vector $\mathbf{v}_t$ given the policy described above. *(No code needed — express it in terms of $a$ and $1-a$.)*

The function `simulate_affine(A, x_0, n, seed)` simulates the affine update $\mathbf{x}_{t+1} = A\mathbf{x}_t + \mathbf{v}_t$ for `n` steps and returns $\mathbf{x}_n$.

In [ ]:
# Run this cell without modification. 
def simulate_affine(A, x_0, n, seed=42):
    rng = np.random.default_rng(seed)
    x = x_0.copy().astype(float)
    for t in range(n):
        a = rng.uniform(0, 1)
        v = np.array([0.005 * a, 0.0, 0.005 * (1 - a), 0.0])
        x = A @ x + v
    return x

In [ ]:
# This cell calls the function and returns x_100_affine. 
x_100_affine = simulate_affine(A_vax, x_0_sird, 100, seed=42)
print('x_100 (vaccination + immigration) =')
print(np.round(x_100_affine.reshape(-1, 1), 4))
print('Total population:', np.round(x_100_affine.sum(), 4))

Run the following cell to plot all four compartments over 300 steps.

In [ ]:
# Run this cell without modification
T_aff = 300
history_aff = np.zeros((4, T_aff+1))
history_aff[:, 0] = x_0_sird
x = x_0_sird.copy().astype(float)
rng_plot = np.random.default_rng(42)
for t in range(T_aff):
    a = rng_plot.uniform(0, 1)
    v = np.array([0.005 * a, 0.0, 0.005 * (1 - a), 0.0])
    x = A_vax @ x + v
    history_aff[:, t+1] = x

fig, ax = plt.subplots(figsize=(9, 5))
for i in range(4):
    ax.plot(np.arange(T_aff+1), history_aff[i], color=colors_sird[i], lw=2, label=labels_sird[i])
ax.set_xlabel('Time step $t$')
ax.set_ylabel('Population (fraction of initial)')
ax.set_title('Vaccination + Immigration Model (seed=42)')
ax.legend(fontsize=9)
ax.set_xlim(0, T_aff)
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

**Exercise 9.** Use `simulate_affine()` with a seed of `11` to calculate the total population at $t = 100$ for the model that incorporates vaccination and immigration using the $x_0$ from the in-class SIRD model. Given the rate of immigration, why does this answer make sense? Did we need to use `simulate_affine()` for this?  

In [ ]:
# Your code here.

**Exercise 10.** Use `simulate_affine()` with the seed set to `11` to predict the *percentage of the entire population* (with vaccination and immigration that will be recovered at $t = 1000$. 

In [ ]:
# Your code here.

## 8.5 Optional: Loop vs. Recursion

*This exercise is intended for students interested in the connection between mathematics and programming. It is not graded.*

The function `simulate_affine` above uses a **loop** to iterate forward through time: we start at $t=0$ and repeatedly apply the update rule until we reach $t=n$.

But notice that the update rule $\mathbf{x}_n = A\mathbf{x}_{n-1} + \mathbf{v}_{n-1}$ is naturally **self-referential**: $\mathbf{x}_n$ is defined in terms of $\mathbf{x}_{n-1}$, which is defined in terms of $\mathbf{x}_{n-2}$, and so on, all the way back to the base case $\mathbf{x}_0$. This structure lends itself to a **recursive** implementation.

A recursive function is one that calls itself. The key ingredients are:
- A **base case** that returns an answer directly (here: $n = 0$, return $\mathbf{x}_0$).
- A **recursive case** that reduces the problem by one step (here: compute $\mathbf{x}_{n-1}$ recursively, then apply the update).

One subtlety: a recursive function unwinds from the inside out — it reaches the base case first, then builds back up. This means we cannot draw random values inside the recursion in order (the draws would happen in reverse). Instead, we generate all $n$ random values up front using the seeded generator, store them in a list, and index into that list at each step.

**Exercise.** Finish the recursive version of `simulate_affine` called `simulate_affine_recursive(A, x_0, n, seed)`. Verify that it gives the same result as the loop version for small $n$ (e.g., $n = 10$ or $n = 50$).

*Note:* Python's default recursion limit is about 1000 calls. For large $n$, the recursive version will raise a `RecursionError`. This is a fundamental limitation of Python (not of recursion as a concept) — languages like Haskell handle deep recursion efficiently. For practical use, the loop version is always preferred in Python but this is a good opportunity to practice recursive programming. See if you can get `simulate_affine_recursive()` to give you the same results as `affine_recursive()`!

In [ ]:
# Your code here
def simulate_affine_recursive(A, x_0, n, seed=42):
    rng    = np.random.default_rng(seed)
    a_vals = [rng.uniform(0, 1) for _ in range(n)]

    def recurse(t):
        # Finish this!

    return recurse(n)

# Verify against loop version
n_test = 50
loop_result = simulate_affine(A_vax, x_0_sird, n_test, seed=42)
rec_result  = simulate_affine_recursive(A_vax, x_0_sird, n_test, seed=42)

print('Loop result:     ', np.round(loop_result, 4))
print('Recursive result:', np.round(rec_result, 4))
print('Match:', np.allclose(loop_result, rec_result))